# Arabic Cursive OCR Benchmark — Unattended Sweep

Benchmarks several local models on **Muharaf** lines, **Run all and walk
away**. After every (model, phase) it rebuilds the leaderboard, mirrors
results to Drive, and pushes them to the repo — so a disconnect never
loses more than the model in flight.

**Resumable:** re-running skips anything already in
`results/sweep_manifest.json`. Safe to Run all again after a timeout.

**Before running:** `Runtime -> T4 GPU`; add `GH_TOKEN` (write-scoped) and
`HF_TOKEN` secrets (see `notebooks/README.md`).

## 1. GPU (falls back to CPU)

In [ ]:
import torch
HAS_GPU = torch.cuda.is_available()
if HAS_GPU:
    print('GPU:', torch.cuda.get_device_name(0))
    !nvidia-smi --query-gpu=memory.total,memory.used --format=csv
else:
    print('NO GPU — will run on CPU (much slower; big 4-bit models are skipped).')
    print('For a real run: Runtime -> Change runtime type -> T4 GPU.')

## 2. Repo

In [ ]:
# Bulletproof checkout: works whether the repo is missing, at the wrong
# path, or has local edits. Always ends up at the latest origin/main.
import os
REPO = '/content/arabic-ocr-benchmark'
token = None
try:
    from google.colab import userdata
    token = userdata.get('GH_TOKEN')
except Exception as e:
    print('no GH_TOKEN secret:', e)
url = f"https://{(token + '@') if token else ''}github.com/rabeeh97/arabic-ocr-benchmark.git"
if os.path.isdir(REPO + '/.git'):
    os.system(f'git -C {REPO} fetch --depth 1 origin main && git -C {REPO} reset --hard origin/main')
    # drop untracked leftovers (e.g. a stale sweep_manifest.json) so the
    # manifest is always exactly what's in the repo. Keeps gitignored _cache.
    os.system(f'git -C {REPO} clean -fd -- results')
else:
    os.system(f'rm -rf {REPO} && git clone --depth 1 {url} {REPO}')
os.chdir(REPO)
print(os.popen('git log --oneline -3').read())

## 3. Install + caches

In [ ]:
# Keep all downloads on Colab's own scratch disk (ephemeral, ~78 GB) --
# never the local machine. Nothing here touches your computer.
import os
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
# Optional HF_TOKEN secret: lifts anonymous rate limits and speeds the
# Muharaf download (and is required for gated models like PaliGemma's base).
try:
    from google.colab import userdata
    _hf = userdata.get('HF_TOKEN')
    if _hf:
        os.environ['HF_TOKEN'] = _hf
        print('HF_TOKEN set')
except Exception:
    pass

In [ ]:
!pip install -q -e '.[local]'

## 4. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Config

`PARTIAL_LIMIT` runs a quick pass on every model first (fast feedback);
with `RUN_FULL = True` it then does the full 1,334-line pass on each.
Both phases are checkpointed independently, so a small partial limit
just means a bit of repeated cheap work if you interrupt.

The default set is the four models that run in fp16 on a free T4. The
7B models (`qwen2.5-vl-7b`, `ain-7b`) are **excluded** — 4-bit loading
staged through system RAM keeps OOM-ing the free tier. Run those
separately on a bigger GPU. On CPU (no GPU) the run shrinks to a tiny
partial-only pass — enough to prove the pipeline, not a real benchmark.

In [ ]:
MODELS = [
    'qari-ocr-0.2',    # Arabic OCR, 2B, raw-text focused (lead)
    'qari-ocr-0.3',    # Arabic OCR, 2B, structure-first (emits markup)
    'qwen2-vl-2b',     # untuned base — anchor
    'qwen2.5-vl-3b',   # general VLM, 3B
]
DATASET = 'muharaf_lines'
PARTIAL_LIMIT = 25       # quick pass per model
RUN_FULL = True          # then the whole 1,334-line test split

if not HAS_GPU:
    PARTIAL_LIMIT = min(PARTIAL_LIMIT, 10)
    RUN_FULL = False
    print(f'CPU mode -> PARTIAL_LIMIT={PARTIAL_LIMIT}, RUN_FULL=False')

## 6. Run the sweep — this is the long one; leave it running

In [ ]:
import shutil, subprocess, pathlib
from arabic_ocr_bench.sweep import run_sweep
from arabic_ocr_bench.samples import export_run_samples

DRIVE = pathlib.Path('/content/drive/MyDrive/arabic-ocr-bench')
DRIVE_OK = DRIVE.parent.exists()   # is Drive mounted?

_DS = {}
def _get_image_fn(dataset):
    # row-index -> PIL image, for the sample export (Muharaf lines only for now)
    if dataset not in _DS:
        from datasets import load_dataset
        _DS[dataset] = load_dataset('aamijar/muharaf-public', split='test')
    ds = _DS[dataset]
    return lambda i: ds[int(i)]['image']

def _git(*a):
    return subprocess.run(['git', *a], capture_output=True, text=True)

def _mirror_to_drive(run_dir):
    if not DRIVE_OK:
        return
    (DRIVE / 'results').mkdir(parents=True, exist_ok=True)
    if pathlib.Path('results/_cache').is_dir():
        shutil.copytree('results/_cache', DRIVE / '_cache', dirs_exist_ok=True)
    shutil.copytree(run_dir, DRIVE / 'results' / pathlib.Path(run_dir).name, dirs_exist_ok=True)
    for f in ('sweep_manifest.json', 'leaderboard.html', 'leaderboard.json'):
        if pathlib.Path('results', f).is_file():
            shutil.copy2(pathlib.Path('results', f), DRIVE / 'results' / f)

def checkpoint(entry, run_dir):
    # Called by run_sweep AFTER a phase is scored. Persist the results here.
    # RAISE if they don't make it to the repo -> the phase is rolled back and
    # retried next run (so a 'done' entry always means 'in the repo').
    run_dir = str(run_dir)
    try:
        export_run_samples(run_dir, get_image=_get_image_fn(entry['dataset']), k=3)
    except Exception as e:
        print('   sample export failed:', e)
    subprocess.run(['python', 'scripts/make_leaderboard.py', 'results',
                    '-o', 'results/leaderboard.html'])
    _mirror_to_drive(run_dir)
    if not token:
        print('   no GH_TOKEN -> Drive only (this counts as persisted)')
        return
    _git('config', 'user.email', 'colab@arabic-ocr-bench')
    _git('config', 'user.name', 'colab-sweep')
    _git('add', '-f', run_dir, 'results/sweep_manifest.json',
         'results/leaderboard.html', 'results/leaderboard.json')
    _git('commit', '-m', f"sweep: {entry['model']} {entry['phase']} "
         f"CER-light {entry['cer_micro_light']*100:.1f}% n={entry['n']}")
    _git('pull', '--rebase', '--autostash', 'origin', 'main')
    p = _git('push', 'origin', 'HEAD:main')
    if p.returncode != 0:
        raise RuntimeError('git push failed: ' + (p.stderr.strip()[:300] or 'unknown'))
    print('   push: ok')

# The manifest is git-tracked (comes from the fresh checkout) — it is the
# single source of truth for what's done. Only restore the per-line cache
# from Drive, to make a re-running phase fast.
if DRIVE_OK and (DRIVE / '_cache').is_dir():
    shutil.copytree(DRIVE / '_cache', 'results/_cache', dirs_exist_ok=True)
    print('restored line cache from Drive')

run_sweep(
    models=MODELS, dataset=DATASET,
    partial_limit=PARTIAL_LIMIT, run_full=RUN_FULL,
    manifest_path='results/sweep_manifest.json', results_dir='results',
    on_progress=print, on_done=checkpoint,
)
print('\n=== SWEEP COMPLETE ===')

## 7. Final leaderboard

In [ ]:
!python scripts/make_leaderboard.py results -o results/leaderboard.html
from IPython.display import HTML
HTML(open('results/leaderboard.html').read())

## Notes

- **Resume after a timeout:** just Run all again. Completed (model, phase)
  pairs are skipped; a partially-done model restarts but its already-scored
  lines come from the cache (restored from Drive), so it's quick.
- **Force a re-run** of one model: delete its entries from
  `results/sweep_manifest.json` (and optionally its `results/<run_id>/`).
- **Add models:** edit `MODELS` and Run all — only the new ones execute.
- Hosted models (Gemini/GPT/Claude): `02_colab_hosted_apis.ipynb`.